## Cleaning of the Huang Award Data

In [1]:
import pandas as pd

# Load your scraped data
df = pd.read_csv("huang_awards_complete_sel.csv")

print(f"Total awards: {len(df)}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"Unique conferences: {df['conference'].nunique()}")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nSample:")
print(df.head(10))


Total awards: 1507
Year range: 1996 - 2023
Unique conferences: 32

Missing values:
year           0
conference     0
paper_title    0
paper_url      0
authors        0
dtype: int64

Sample:
   year conference                                        paper_title  \
0  2023       AAAI  Misspecification in Inverse Reinforcement Lear...   
1  2023        ACL  Do Androids Laugh at Electric Sheep? Humor "Un...   
2  2023        ACL  What the DAAM: Interpreting Stable Diffusion U...   
3  2023        ACL  From Pretraining Data to Language Models to Do...   
4  2023        CHI  Breaking Out of the Ivory Tower: A Large-scale...   
5  2023        CHI  Changes in Research Ethics, Openness, and Tran...   
6  2023        CHI  ChartDetective: Easy and Accurate Interactive ...   
7  2023        CHI  CiteSee: Augmenting Citations in Scientific Pa...   
8  2023        CHI  Collaborating Across Realities: Analytical Len...   
9  2023        CHI  Contestable Camera Cars: A Speculative Design ...   

      

## Cleaning

In [2]:
# 1. Remove rows with missing critical fields
df_clean = df.dropna(subset=['year', 'conference', 'paper_title']).copy()

# 2. Remove duplicates (same paper title + year)
df_clean = df_clean.drop_duplicates(subset=['paper_title', 'year'], keep='first')

# 3. Clean paper titles (remove extra whitespace)
df_clean['paper_title'] = df_clean['paper_title'].str.strip()

## Filtering

In [3]:
# 4. Filter to your pilot scope (2010-2020, selected conferences)
PILOT_CONFERENCES = ['CHI', 'OSDI', 'SIGMOD', 'NeurIPS', 'ICML', 'PLDI', 'SOSP', 'ICSE']
START_YEAR = 2000
END_YEAR = 2018

df_pilot = df_clean[
    (df_clean['conference'].isin(PILOT_CONFERENCES)) &
    (df_clean['year'].astype(int) >= START_YEAR) &
    (df_clean['year'].astype(int) <= END_YEAR)
].copy()

print(f"\n{'='*60}")
print(f"After cleaning:")
print(f"  Total: {len(df_clean)} awards")
print(f"  Pilot (2000-2018, {len(PILOT_CONFERENCES)} conferences): {len(df_pilot)} awards")
print(f"\nPilot breakdown:")
print(df_pilot.groupby('conference')['year'].agg(['count', 'min', 'max']))

# Save cleaned versions
df_clean.to_csv("huang_awards_cleaned.csv", index=False)
df_pilot.to_csv("huang_awards_pilot.csv", index=False)


After cleaning:
  Total: 1506 awards
  Pilot (2000-2018, 8 conferences): 411 awards

Pilot breakdown:
            count   min   max
conference                   
CHI           192  2005  2018
ICML           19  2005  2018
ICSE           88  2003  2018
NeurIPS        15  2013  2018
OSDI           21  2000  2018
PLDI           32  2000  2018
SIGMOD         19  2000  2018
SOSP           25  2001  2017
